* kaggle에서 테스트한 예제
* GPU 이용

# 준비

In [1]:
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 데이터

In [3]:
train_data = datasets.CIFAR10("data", train=True, download=True)
test_data = datasets.CIFAR10("data", train=False, download=True)
classes = train_data.classes

100%|██████████| 170M/170M [19:37<00:00, 145kB/s] 


# 전처리

In [4]:
weights = models.ResNet18_Weights.DEFAULT
transform = weights.transforms()          # 224×224로 키우고, 학습 때와 같은 기준으로 스케일링
train_data.transform = transform
test_data.transform = transform

train_loader = DataLoader(Subset(train_data, range(10000)), batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(Subset(test_data, range(2000)), batch_size=128, num_workers=2)

# 모델

In [5]:
model = models.resnet18(weights=weights)

for param in model.parameters():          # 앞부분은 학습하지 않는다 (얼리기)
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 10)   # 마지막 층을 10종류용으로 교체
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 숫자 {total:,}개 중 학습할 숫자 {trainable:,}개")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


전체 숫자 11,181,642개 중 학습할 숫자 5,130개


# 학습

In [8]:
EPOCHS = 20
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, EPOCHS + 1):
    model.eval()                      # 이 예제만 다르다: 얼린 층은 예측 모드로 둔다
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)              # 1. 순전파
        loss = loss_fn(pred, yb)      # 2. 손실
        optimizer.zero_grad()         #    지난번 계산 지우기
        loss.backward()               # 3. 역전파
        optimizer.step()              # 4. 수정
        total_loss += loss.item()
    # 출력을 EPOCHS에 상관없이 무조건 5번 하기 위한 코드
    if epoch % (EPOCHS // 5) == 0:
        print(f"에포크 {epoch:3}: 손실 {total_loss / len(train_loader):.3f}")

에포크   4: 손실 0.387
에포크   8: 손실 0.349
에포크  12: 손실 0.328
에포크  16: 손실 0.304
에포크  20: 손실 0.296


# 평가

In [9]:
model.eval()
preds, answers = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds.append(model(xb.to(device)).cpu())
        answers.append(yb)
preds, answers = torch.cat(preds), torch.cat(answers)

labels = preds.argmax(dim=1)
print(f"정확도: {accuracy_score(answers, labels):.3f}")

정확도: 0.839


# 저장하고 예측

In [12]:
Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/cifar_resnet18.pt")

for i in range(10):
    image, answer = test_data[i]
    with torch.no_grad():
        probs = torch.softmax(model(image.unsqueeze(0).to(device)), dim=1)[0]
    
    print(f"예측: {classes[probs.argmax()]} ({probs.max():.3f}) / 정답: {classes[answer]}")

예측: cat (0.893) / 정답: cat
예측: ship (0.993) / 정답: ship
예측: ship (0.988) / 정답: ship
예측: ship (0.555) / 정답: airplane
예측: frog (1.000) / 정답: frog
예측: frog (0.516) / 정답: frog
예측: automobile (0.722) / 정답: automobile
예측: frog (0.986) / 정답: frog
예측: cat (0.966) / 정답: cat
예측: automobile (0.954) / 정답: automobile
